In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revelead it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
        }
    )

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My favourite colour is green
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_NsSenlTE3Z4sZbfgwGClQn5J)
 Call ID: call_NsSenlTE3Z4sZbfgwGClQn5J
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Done. I’ve updated your favourite colour to green.

Would you like me to tailor future interactions around green, or save any other preferences (like themes, fonts, or prompts)? I can also generate a green-themed color palette if you’d like.


In [7]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My favourite colour is green
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_NsSenlTE3Z4sZbfgwGClQn5J)
 Call ID: call_NsSenlTE3Z4sZbfgwGClQn5J
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Done. I’ve updated your favourite colour to green.

Would you like me to tailor future interactions around green, or save any other preferences (like themes, fonts, or prompts)? I can also generate a green-themed color palette if you’d like.
================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Hell

## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My favourite colour is green
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_G6DtX3pv3u6r08CLFmK0B63Z)
 Call ID: call_G6DtX3pv3u6r08CLFmK0B63Z
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Great! I’ve updated your favourite colour to green.

Would you like me to remember anything else or tell you what I have on file about your preferences? I can also confirm your current favourite colour if you’d like.


In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My favourite colour is green
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_G6DtX3pv3u6r08CLFmK0B63Z)
 Call ID: call_G6DtX3pv3u6r08CLFmK0B63Z
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Great! I’ve updated your favourite colour to green.

Would you like me to remember anything else or tell you what I have on file about your preferences? I can also confirm your current favourite colour if you’d like.
================================ Human Message =================================

What's my favourite colour?
================================== Ai Message ==================================
Tool Calls:
  read_fav